In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Zadávání možností Sampleru

<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut za použití následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Možnosti můžeš použít k přizpůsobení primitivu Sampler. Tato část se zaměřuje na to, jak zadávat možnosti primitivů Qiskit Runtime. Ačkoli je rozhraní metody `run()` primitivů společné pro všechny implementace, jejich možnosti nikoli. Prostuduj si příslušné reference API pro informace o možnostech [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) a [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Nastavení možností Sampleru
Možnosti můžeš nastavit při inicializaci Sampleru, po inicializaci Sampleru nebo je aktualizovat po inicializaci. Pokyny k použití těchto technik najdeš v tématu [Úvod do možností](/guides/runtime-options-overview#options-precedence).

Navíc můžeš nastavit hodnotu `shots` v metodě `run()`, jak je popsáno v následující části.
<span id="run-method"></span>
### Metoda Run()
Jediné hodnoty, které můžeš předat do `run()`, jsou ty definované v rozhraní, tedy `shots`. Tímto se přepíše hodnota nastavená pro `default_shots` pro aktuální spuštění.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

<Admonition type="note">
Although shots specified in the PUB and in `run` have higher precedence, the job fails if `twirling` is enabled and the product of `num_randomizations` and `shots_per_randomization` is smaller than the `shots` value. In this scenario, `SamplerV2` is unable to allocate the shots among the specified `num_randomizations`.
</Admonition>

Example:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden
# if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Speciální případy
<span id="shots"></span>
#### Shots

Metoda `SamplerV2.run` přijímá dva argumenty: seznam PUBů, z nichž každý může specifikovat PUB-specifickou hodnotu shots, a klíčový argument shots. Tyto hodnoty shots jsou součástí rozhraní pro spouštění Sampleru a jsou nezávislé na možnostech Sampleru Runtime. Mají přednost před hodnotami zadanými jako možnosti, aby byla zachována shoda s abstrakcí Sampleru.

Pokud však `shots` není zadáno žádným PUBem ani v klíčovém argumentu run (nebo pokud jsou všechny `None`), použije se hodnota shots z možností, zejména `default_shots`.

Shrnutí pořadí priority pro zadávání shots v Sampleru pro libovolný PUB:

1. Pokud PUB specifikuje shots, použij tuto hodnotu.
2. Pokud je klíčový argument `shots` zadán v `run`, použij tuto hodnotu.
4. Pokud je povoleno `twirling` (ve výchozím nastavení True), použije se součin `num_randomizations` a `shots_per_randomization`, zadaný jako [možnosti `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
5. Pokud je zadáno `sampler.options.default_shots`, použij tuto hodnotu.

Pokud jsou tedy shots zadány na všech možných místech, použije se to s nejvyšší prioritou (shots zadané v PUBu).

> **Note:** Přestože shots zadané v PUBu a v `run` mají vyšší prioritu, úloha selže, pokud je povoleno `twirling` a součin `num_randomizations` a `shots_per_randomization` je menší než hodnota `shots`. V tomto scénáři není `SamplerV2` schopen rozdělit shots mezi zadané `num_randomizations`.

Příklad: